In [11]:
import sqlite3
import pandas as pd

In [12]:
conn = sqlite3.connect("backup_analytics.db")

In [21]:
print(os.listdir("data/raw"))

['backup_jobs.csv', 'customers.csv', 'customers_df.csv', 'recovery_logs.csv', 'security_alerts.csv', 'storage_usage.csv']


In [22]:
import pandas as pd

customers_df = pd.read_csv("data/raw/customers.csv")
backup_jobs_df = pd.read_csv("data/raw/backup_jobs.csv")
storage_usage_df = pd.read_csv("data/raw/storage_usage.csv")
security_alerts_df = pd.read_csv("data/raw/security_alerts.csv")
recovery_logs_df = pd.read_csv("data/raw/recovery_logs.csv")

In [23]:
print(os.listdir("data"))

['raw']


In [24]:
import os
print(os.listdir("data/raw"))


['backup_jobs.csv', 'customers.csv', 'customers_df.csv', 'recovery_logs.csv', 'security_alerts.csv', 'storage_usage.csv']


In [25]:
import pandas as pd

customers_df = pd.read_csv("data/raw/customers.csv")
backup_jobs_df = pd.read_csv("data/raw/backup_jobs.csv")
storage_usage_df = pd.read_csv("data/raw/storage_usage.csv")
security_alerts_df = pd.read_csv("data/raw/security_alerts.csv")
recovery_logs_df = pd.read_csv("data/raw/recovery_logs.csv")

In [26]:
customers_df.head()

,customer_id,company_name,industry,cloud_provider,company_size,region,subscription_plan
0,1,Leach-Dunn,Finance,AWS,Large,Europe,Standard
1,2,Knight Inc,Healthcare,Google Cloud,Small,North America,Enterprise
2,3,Garcia-Cohen,Technology,AWS,Small,North America,Standard
3,4,Hayes-Marsh,Healthcare,Google Cloud,Small,Europe,Enterprise
4,5,Mason Group,Manufacturing,Azure,Medium,South America,Enterprise


In [27]:
customers_df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

backup_jobs_df.to_sql(
    "backup_jobs",
    conn,
    if_exists="replace",
    index=False
)

storage_usage_df.to_sql(
    "storage_usage",
    conn,
    if_exists="replace",
    index=False
)

security_alerts_df.to_sql(
    "security_alerts",
    conn,
    if_exists="replace",
    index=False
)

recovery_logs_df.to_sql(
    "recovery_logs",
    conn,
    if_exists="replace",
    index=False
)

3208

In [28]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,customers
1,backup_jobs
2,storage_usage
3,security_alerts
4,recovery_logs


In [32]:
#Which Cloud Provider Has the Highest Failure Rate?
query = """
SELECT
    cloud_provider,
    COUNT(*) AS total_jobs,
    SUM(CASE WHEN backup_status = 'Failed' THEN 1 ELSE 0 END) AS failed_jobs,
    ROUND(
        100.0 * SUM(CASE WHEN backup_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS failure_rate
FROM backup_jobs
GROUP BY cloud_provider
ORDER BY failure_rate DESC;
"""
pd.read_sql(query, conn)

,cloud_provider,total_jobs,failed_jobs,failure_rate
0,Google Cloud,16400,1113,6.79
1,AWS,17050,1077,6.32
2,Azure,16550,1018,6.15


In [33]:
#Top 10 Customers by Storage Cost
query = """
SELECT
    c.company_name,
    ROUND(SUM(b.storage_cost_usd),2) AS total_storage_cost
FROM customers c
JOIN backup_jobs b
ON c.customer_id = b.customer_id
GROUP BY c.company_name
ORDER BY total_storage_cost DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,company_name,total_storage_cost
0,Williams Group,16697.87
1,Marshall LLC,16534.97
2,Khan PLC,16474.25
3,Williams and Sons,15577.37
4,Brown Inc,12811.84
5,Smith PLC,12571.09
6,Anderson Ltd,11729.93
7,Smith LLC,11555.30
8,Campbell and Sons,11330.53
9,Davis and Sons,9944.73


In [34]:
#Average Backup Size by Company Size
query = """
SELECT
    c.company_size,
    ROUND(AVG(b.backup_size_gb),2) AS average_backup_size
FROM customers c
JOIN backup_jobs b
ON c.customer_id = b.customer_id
GROUP BY c.company_size
ORDER BY average_backup_size;
"""

pd.read_sql(query, conn)

,company_size,average_backup_size
0,Small,175.18
1,Medium,901.54
2,Large,3239.62
3,Enterprise,8495.13


In [35]:
#Which Region Generates the Highest Storage Cost?
query = """
SELECT
    region,
    ROUND(SUM(storage_cost_usd),2) AS total_storage_cost
FROM backup_jobs
GROUP BY region
ORDER BY total_storage_cost DESC;
"""

pd.read_sql(query, conn)

,region,total_storage_cost
0,Asia Pacific,853349.47
1,South America,844417.68
2,North America,839171.12
3,Europe,791864.99


In [36]:
#Top 10 Customers by Failed Backups
query = """
SELECT
    c.company_name,
    COUNT(*) AS failed_backups
FROM customers c
JOIN backup_jobs b
ON c.customer_id = b.customer_id
WHERE b.backup_status='Failed'
GROUP BY c.company_name
ORDER BY failed_backups DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,company_name,failed_backups
0,Brown Inc,14
1,Smith LLC,13
2,Brooks-Payne,13
3,Stewart-Sanders,12
4,Smith PLC,12
5,Myers-Mays,12
6,Marshall LLC,12
7,"Dunn, Baker and Brooks",12
8,Downs-Miller,12
9,Williams Group,11


In [37]:
#Recovery Success Rate
query = """
SELECT
    recovery_status,
    COUNT(*) AS total
FROM recovery_logs
GROUP BY recovery_status;
"""

pd.read_sql(query, conn)

,recovery_status,total
0,Failed,1088
1,Partially Successful,1058
2,Successful,1062


In [38]:
#Security Alerts by Severity
query = """
SELECT
    severity,
    COUNT(*) AS total_alerts
FROM security_alerts
GROUP BY severity
ORDER BY total_alerts DESC;
"""

pd.read_sql(query, conn)

,severity,total_alerts
0,Low,862
1,Medium,832
2,High,801


In [39]:
#Backup Jobs by Company Size
query = """
SELECT
    c.company_size,
    COUNT(*) AS total_jobs
FROM customers c
JOIN backup_jobs b
ON c.customer_id=b.customer_id
GROUP BY c.company_size;
"""

pd.read_sql(query, conn)

,company_size,total_jobs
0,Enterprise,13250
1,Large,12400
2,Medium,13000
3,Small,11350
